In [1]:
# ==== ZERO-SHOT TEST (no fine-tuning) ====
# Baseline: Pretrained AST/PaSST with a randomly initialized 2-class head

import os
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments
)
import evaluate
import glob
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --------------------
# Paths
# --------------------
MODEL_ID   = r"D:\Thesis\Song\TrainClassification\passt-light-finetuned\checkpoint-7158"  # <-- your finetuned model
OUTPUT_DIR = "./passt-finetuned-eval"                 # new output folder
TEST_DIR   = "Speech"  # adjust if needed

# --------------------
# Load extractor & model
# --------------------
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

# Load pretrained model, force new binary head (random init)
zero_shot_model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
)

TARGET_SR = feature_extractor.sampling_rate  # 32000 Hz

# --------------------
# Dataset
# --------------------
class PaSSTDataset(Dataset):
    def __init__(self, root_dir, feature_extractor, target_sr):
        self.files, self.labels = [], []
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr
        min_len = int(0.025 * target_sr)  # 25 ms minimum

        # search recursively for .wav files
        all_files = glob.glob(os.path.join(root_dir, "**", "*.wav"), recursive=True)
        all_files += glob.glob(os.path.join(root_dir, "**", "*.WAV"), recursive=True)

        for path in sorted(all_files):
            # assign label based on parent folder
            parent = os.path.basename(os.path.dirname(path)).lower()
            if parent == "real":
                label = 0
            elif parent == "fake":
                label = 1
            else:
                continue  # skip anything outside Real/Fake

            # check length
            waveform, sr = torchaudio.load(path)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            if sr != target_sr:
                waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            waveform = waveform.squeeze()
            if waveform.shape[-1] >= min_len:
                self.files.append(path)
                self.labels.append(label)

        print(f"✅ Loaded {len(self.files)} test files from {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        waveform = waveform.squeeze().numpy()

        inputs = self.feature_extractor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=True
        )

        item = {
            "input_values": inputs["input_values"].squeeze(0).numpy(),
            "labels": np.int64(label)
        }
        if "attention_mask" in inputs:
            item["attention_mask"] = inputs["attention_mask"].squeeze(0).numpy()

        return item


test_dataset = PaSSTDataset(TEST_DIR, feature_extractor, TARGET_SR)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "auroc": roc_metric.compute(prediction_scores=probs, references=labels)["roc_auc"]
    }

# --------------------
# Trainer (evaluate only)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_eval_batch_size=64,
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=zero_shot_model,
    args=args,
    eval_dataset=test_dataset,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

# --------------------
# Run evaluation
# --------------------
metrics = trainer.evaluate(test_dataset)
print("\n=== Zero-Shot Test Metrics ===")
for k, v in metrics.items():
    if k.startswith("eval_"):
        print(f"{k[5:]}: {v:.4f}")

# --------------------
# Detailed Classification Report
# --------------------
preds = trainer.predict(test_dataset)
logits, labels = preds.predictions, preds.label_ids
y_pred = np.argmax(logits, axis=-1)
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

target_names = ["Real (0)", "Fake (1)"]
rep_str = classification_report(labels, y_pred, target_names=target_names, digits=4)
print("\n=== Classification Report (Zero-Shot) ===")
print(rep_str)

rep_dict = classification_report(labels, y_pred, target_names=target_names, output_dict=True)
rep_df = pd.DataFrame(rep_dict).transpose()

cm = confusion_matrix(labels, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm,
    index=["True Real (0)", "True Fake (1)"],
    columns=["Pred Real (0)", "Pred Fake (1)"]
)
print("\n=== Confusion Matrix (Zero-Shot) ===")
print(cm_df)

# Save reports
os.makedirs(OUTPUT_DIR, exist_ok=True)
rep_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report_test.csv"), index=True)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_test.csv"), index=True)

per_file_df = pd.DataFrame({
    "Filename": [os.path.basename(f) for f in test_dataset.files],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in labels],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": probs
})
per_file_df.to_csv(os.path.join(OUTPUT_DIR, "per_file_predictions.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "classification_report_test.txt"), "w", encoding="utf-8") as f:
    f.write(rep_str)

print("\n✅ Saved detailed reports into", OUTPUT_DIR)



✅ Loaded 2544 test files from Speech


C:\Users\j3n50\AppData\Local\Temp\ipykernel_27392\838309336.py:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



=== Zero-Shot Test Metrics ===
loss: 3.0296
model_preparation_time: 0.0027
accuracy: 0.6344
f1_macro: 0.5704
auroc: 0.9235
runtime: 51.3404
samples_per_second: 49.5520
steps_per_second: 0.7790

=== Classification Report (Zero-Shot) ===
              precision    recall  f1-score   support

    Real (0)     0.5836    0.9969    0.7362      1302
    Fake (1)     0.9875    0.2544    0.4046      1242

    accuracy                         0.6344      2544
   macro avg     0.7856    0.6257    0.5704      2544
weighted avg     0.7808    0.6344    0.5743      2544


=== Confusion Matrix (Zero-Shot) ===
               Pred Real (0)  Pred Fake (1)
True Real (0)           1298              4
True Fake (1)            926            316

✅ Saved detailed reports into ./passt-finetuned-eval


In [4]:
# ==== ZERO-SHOT TEST (no fine-tuning) ====
# Baseline: Pretrained AST/PaSST with a randomly initialized 2-class head

import os
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments
)
import evaluate
import glob
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --------------------
# Paths
# --------------------
MODEL_ID   = r"D:\Thesis\Song\TrainClassification\ast-light-finetuned\checkpoint-4000"  # <-- your finetuned model
OUTPUT_DIR = "./passt-finetuned-eval"                 # new output folder
TEST_DIR   = "Speech"  # adjust if needed

# --------------------
# Load extractor & model
# --------------------
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

# Load pretrained model, force new binary head (random init)
zero_shot_model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
)

TARGET_SR = feature_extractor.sampling_rate  # 32000 Hz

# --------------------
# Dataset
# --------------------
class PaSSTDataset(Dataset):
    def __init__(self, root_dir, feature_extractor, target_sr):
        self.files, self.labels = [], []
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr
        min_len = int(0.025 * target_sr)  # 25 ms minimum

        # search recursively for .wav files
        all_files = glob.glob(os.path.join(root_dir, "**", "*.wav"), recursive=True)
        all_files += glob.glob(os.path.join(root_dir, "**", "*.WAV"), recursive=True)

        for path in sorted(all_files):
            # assign label based on parent folder
            parent = os.path.basename(os.path.dirname(path)).lower()
            if parent == "real":
                label = 0
            elif parent == "fake":
                label = 1
            else:
                continue  # skip anything outside Real/Fake

            # check length
            waveform, sr = torchaudio.load(path)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            if sr != target_sr:
                waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            waveform = waveform.squeeze()
            if waveform.shape[-1] >= min_len:
                self.files.append(path)
                self.labels.append(label)

        print(f"✅ Loaded {len(self.files)} test files from {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        waveform = waveform.squeeze().numpy()

        inputs = self.feature_extractor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=True
        )

        item = {
            "input_values": inputs["input_values"].squeeze(0).numpy(),
            "labels": np.int64(label)
        }
        if "attention_mask" in inputs:
            item["attention_mask"] = inputs["attention_mask"].squeeze(0).numpy()

        return item


test_dataset = PaSSTDataset(TEST_DIR, feature_extractor, TARGET_SR)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "auroc": roc_metric.compute(prediction_scores=probs, references=labels)["roc_auc"]
    }

# --------------------
# Trainer (evaluate only)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_eval_batch_size=64,
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=zero_shot_model,
    args=args,
    eval_dataset=test_dataset,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

# --------------------
# Run evaluation
# --------------------
metrics = trainer.evaluate(test_dataset)
print("\n=== Zero-Shot Test Metrics ===")
for k, v in metrics.items():
    if k.startswith("eval_"):
        print(f"{k[5:]}: {v:.4f}")

# --------------------
# Detailed Classification Report
# --------------------
preds = trainer.predict(test_dataset)
logits, labels = preds.predictions, preds.label_ids
y_pred = np.argmax(logits, axis=-1)
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

target_names = ["Real (0)", "Fake (1)"]
rep_str = classification_report(labels, y_pred, target_names=target_names, digits=4)
print("\n=== Classification Report (Zero-Shot) ===")
print(rep_str)

rep_dict = classification_report(labels, y_pred, target_names=target_names, output_dict=True)
rep_df = pd.DataFrame(rep_dict).transpose()

cm = confusion_matrix(labels, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm,
    index=["True Real (0)", "True Fake (1)"],
    columns=["Pred Real (0)", "Pred Fake (1)"]
)
print("\n=== Confusion Matrix (Zero-Shot) ===")
print(cm_df)

# Save reports
os.makedirs(OUTPUT_DIR, exist_ok=True)
rep_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report_test.csv"), index=True)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_test.csv"), index=True)

per_file_df = pd.DataFrame({
    "Filename": [os.path.basename(f) for f in test_dataset.files],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in labels],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": probs
})
per_file_df.to_csv(os.path.join(OUTPUT_DIR, "per_file_predictions.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "classification_report_test.txt"), "w", encoding="utf-8") as f:
    f.write(rep_str)

print("\n✅ Saved detailed reports into", OUTPUT_DIR)


✅ Loaded 2544 test files from Speech


C:\Users\j3n50\AppData\Local\Temp\ipykernel_32252\2575743475.py:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



=== Zero-Shot Test Metrics ===
loss: 1.9524
model_preparation_time: 0.0030
accuracy: 0.6753
f1_macro: 0.6489
auroc: 0.8764
runtime: 47.3273
samples_per_second: 53.7530
steps_per_second: 0.8450

=== Classification Report (Zero-Shot) ===
              precision    recall  f1-score   support

    Real (0)     0.6227    0.9278    0.7452      1302
    Fake (1)     0.8444    0.4106    0.5525      1242

    accuracy                         0.6753      2544
   macro avg     0.7335    0.6692    0.6489      2544
weighted avg     0.7309    0.6753    0.6512      2544


=== Confusion Matrix (Zero-Shot) ===
               Pred Real (0)  Pred Fake (1)
True Real (0)           1208             94
True Fake (1)            732            510

✅ Saved detailed reports into ./passt-finetuned-eval


In [3]:
# ==== ZERO-SHOT TEST (no fine-tuning) ====
# Baseline: Pretrained AST/PaSST with a randomly initialized 2-class head

import os
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments
)
import evaluate
import glob
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --------------------
# Paths
# --------------------
MODEL_ID   = r"D:\Thesis\Song\TrainClassification\ast-light-finetuned\checkpoint-3500"  # <-- your finetuned model
OUTPUT_DIR = "./passt-finetuned-eval"                 # new output folder
TEST_DIR   = "Speech"  # adjust if needed

# --------------------
# Load extractor & model
# --------------------
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

# Load pretrained model, force new binary head (random init)
zero_shot_model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
)

TARGET_SR = feature_extractor.sampling_rate  # 32000 Hz

# --------------------
# Dataset
# --------------------
class PaSSTDataset(Dataset):
    def __init__(self, root_dir, feature_extractor, target_sr):
        self.files, self.labels = [], []
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr
        min_len = int(0.025 * target_sr)  # 25 ms minimum

        # search recursively for .wav files
        all_files = glob.glob(os.path.join(root_dir, "**", "*.wav"), recursive=True)
        all_files += glob.glob(os.path.join(root_dir, "**", "*.WAV"), recursive=True)

        for path in sorted(all_files):
            # assign label based on parent folder
            parent = os.path.basename(os.path.dirname(path)).lower()
            if parent == "real":
                label = 0
            elif parent == "fake":
                label = 1
            else:
                continue  # skip anything outside Real/Fake

            # check length
            waveform, sr = torchaudio.load(path)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            if sr != target_sr:
                waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            waveform = waveform.squeeze()
            if waveform.shape[-1] >= min_len:
                self.files.append(path)
                self.labels.append(label)

        print(f"✅ Loaded {len(self.files)} test files from {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        waveform = waveform.squeeze().numpy()

        inputs = self.feature_extractor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=True
        )

        item = {
            "input_values": inputs["input_values"].squeeze(0).numpy(),
            "labels": np.int64(label)
        }
        if "attention_mask" in inputs:
            item["attention_mask"] = inputs["attention_mask"].squeeze(0).numpy()

        return item


test_dataset = PaSSTDataset(TEST_DIR, feature_extractor, TARGET_SR)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "auroc": roc_metric.compute(prediction_scores=probs, references=labels)["roc_auc"]
    }

# --------------------
# Trainer (evaluate only)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_eval_batch_size=64,
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=zero_shot_model,
    args=args,
    eval_dataset=test_dataset,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

# --------------------
# Run evaluation
# --------------------
metrics = trainer.evaluate(test_dataset)
print("\n=== Fine-Tuned Test Metrics ===")
for k, v in metrics.items():
    if k.startswith("eval_"):
        print(f"{k[5:]}: {v:.4f}")

# --------------------
# Detailed Classification Report
# --------------------
preds = trainer.predict(test_dataset)
logits, labels = preds.predictions, preds.label_ids
y_pred = np.argmax(logits, axis=-1)
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

target_names = ["Real (0)", "Fake (1)"]
rep_str = classification_report(labels, y_pred, target_names=target_names, digits=4)
print("\n=== Classification Report (Fine-Tuned) ===")
print(rep_str)

rep_dict = classification_report(labels, y_pred, target_names=target_names, output_dict=True)
rep_df = pd.DataFrame(rep_dict).transpose()

cm = confusion_matrix(labels, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm,
    index=["True Real (0)", "True Fake (1)"],
    columns=["Pred Real (0)", "Pred Fake (1)"]
)
print("\n=== Confusion Matrix (Fine-Tuned) ===")
print(cm_df)

# Save reports
os.makedirs(OUTPUT_DIR, exist_ok=True)
rep_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report_test.csv"), index=True)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_test.csv"), index=True)

per_file_df = pd.DataFrame({
    "Filename": [os.path.basename(f) for f in test_dataset.files],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in labels],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": probs
})
per_file_df.to_csv(os.path.join(OUTPUT_DIR, "per_file_predictions.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "classification_report_test.txt"), "w", encoding="utf-8") as f:
    f.write(rep_str)

print("\n✅ Saved detailed reports into", OUTPUT_DIR)


✅ Loaded 2544 test files from Speech


C:\Users\j3n50\AppData\Local\Temp\ipykernel_1784\175532801.py:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



=== Fine-Tuned Test Metrics ===
loss: 1.6971
model_preparation_time: 0.0010
accuracy: 0.7075
f1_macro: 0.6850
auroc: 0.9024
runtime: 47.7490
samples_per_second: 53.2790
steps_per_second: 0.8380

=== Classification Report (Fine-Tuned) ===
              precision    recall  f1-score   support

    Real (0)     0.6452    0.9524    0.7692      1302
    Fake (1)     0.9003    0.4509    0.6009      1242

    accuracy                         0.7075      2544
   macro avg     0.7727    0.7016    0.6850      2544
weighted avg     0.7697    0.7075    0.6870      2544


=== Confusion Matrix (Fine-Tuned) ===
               Pred Real (0)  Pred Fake (1)
True Real (0)           1240             62
True Fake (1)            682            560

✅ Saved detailed reports into ./passt-finetuned-eval
